# 02 · 聚合与分组（GROUP BY / HAVING）

> 上一课是“逐行挑数据”，这一课是“把很多行压成一行做统计”。
> 你在 Pandas 里写过 `df.groupby('region')['sales'].sum()`，SQL 的 `GROUP BY` 是同一个动作：**按某个列把行分门别类，每一类算出一个汇总值**。这一课最大的价值不是语法，而是搞懂“为什么 SELECT 里的非聚合列必须出现在 GROUP BY 里”，以及 WHERE 和 HAVING 的分工。

## 本课地图
1. 聚合函数：COUNT / SUM / AVG / MIN / MAX
2. 重点辨析：COUNT(*) vs COUNT(列) vs COUNT(DISTINCT 列)
3. GROUP BY：分组聚合
4. HAVING：对分组结果再过滤
5. SQL 子句执行顺序（上一课预告，这里彻底讲透）
6. 分组最常见的报错及原因

In [1]:
# 统一环境：连接数据库（绝对路径，可反复运行）
import sqlite3
import pandas as pd

# 两个库都用绝对路径写死，避免“相对路径在哪个工作目录下”的坑
SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
SS_DB  = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\superstore.db"

# 让 DataFrame 打印好看一点（你已经熟悉 Pandas，这里直接拿它当 SQL 的“显示层”）
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# 教学默认连 shop.db；需要 superstore 时再切
conn = sqlite3.connect(SHOP_DB)

def q(sql, conn=conn):
    """执行一条 SQL，把结果表直接返回成 Pandas DataFrame。"""
    return pd.read_sql_query(sql, conn)

print("已连接 shop.db。可用 q('SELECT ...') 跑查询。")
print("shop.db 表：", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

已连接 shop.db。可用 q('SELECT ...') 跑查询。
shop.db 表： ['customers', 'employees', 'order_items', 'orders', 'products']


## 1. 聚合函数：把一列压成一个数

聚合函数把“多行”折叠成“一行”的统计值。没有 GROUP BY 时，它把整张表当成一组。

In [2]:
# 全公司的经营大盘：总销售额、总利润、订单明细行数
# 注意：order_date 在 orders 表，不在 order_items，别拿错表
q('''
SELECT COUNT(*)        AS 明细行数,
       SUM(sales)      AS 总销售额,
       SUM(profit)     AS 总利润,
       AVG(sales)      AS 平均单笔明细销售额
FROM order_items
''')

,明细行数,总销售额,总利润,平均单笔明细销售额
0,6407,25040326.83,8233307.18,3908.27639


In [3]:
# 日期范围属于 orders 表，单独从 orders 取
q('''
SELECT MIN(order_date) AS 最早订单, MAX(order_date) AS 最晚订单
FROM orders
''')

,最早订单,最晚订单
0,2022-01-01,2025-12-28


> 对比 Pandas：这就是 `df.agg(['count','sum','mean','min','max'])`。注意 `AVG` 本质是 `SUM/COUNT`，对大数做平均时小心浮点精度（SQLite 用 REAL，够用）。

## 2. 重点辨析：三种 COUNT

这是面试必问、新手必错。三者结果完全不同：

In [4]:
# COUNT(*)        = 数行数（不管这行内容）
# COUNT(列)       = 数“该列不为 NULL”的行数
# COUNT(DISTINCT 列) = 数“该列有多少个不同的非空值”
q('''
SELECT
  COUNT(*)                  AS 总行数,
  COUNT(manager_id)         AS manager_id非空数,
  COUNT(DISTINCT manager_id) AS manager_id去重后数
FROM employees
''')

,总行数,manager_id非空数,manager_id去重后数
0,54,53,25


employees 有 54 行，其中只有大老板 `manager_id IS NULL`。所以 `COUNT(manager_id)` = 53（跳过了那一个 NULL）。而 `COUNT(*)` 永远是 54。

> 记忆：
> - 想数“这张表/这个结果集有多少行” → 用 `COUNT(*)`；
> - 想数“某列有多少个非空值” → 用 `COUNT(列)`；
> - 想数“有多少个不同客户/商品” → 用 `COUNT(DISTINCT 列)`。

下面这个最有用：一共多少个客户下过单？

In [5]:
# 有多少个不同的客户下过单？（orders 里 2600 行是订单，不是客户）
q('''
SELECT COUNT(DISTINCT customer_id) AS 下单客户数
FROM orders
''')

,下单客户数
0,300


## 3. GROUP BY：分组聚合

`GROUP BY 列` 把表按那列的值切成若干组，每组各自算聚合值。等价 Pandas `groupby(...).agg(...)`。

In [6]:
# 按地区统计：每个地区的订单数、总销售额、平均客单
q('''
SELECT region,
       COUNT(*)               AS 订单数,
       SUM(sales)             AS 总销售额,
       ROUND(AVG(sales), 2)   AS 平均单笔明细额
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY region
ORDER BY 总销售额 DESC
''')

,region,订单数,总销售额,平均单笔明细额
0,华东,1487,6090607.00,4095.90
1,华南,1467,5873532.64,4003.77
2,华中,1409,5364460.57,3807.28
3,西南,1103,4051805.89,3673.44
4,华北,941,3659920.73,3889.40


注意上面：`SELECT` 里 `region` 是“分组键”，其余列都是聚合值。这是铁律——

> **SELECT 中出现的、不在聚合函数里的列，必须全部出现在 GROUP BY 里。**

为什么？因为分组后数据库每组只输出一行，而一个组里可能有几十上百个不同的 `city`，它不知道该给你哪一个。在严格数据库（PostgreSQL）里这会直接报错；SQLite 比较宽松会“随便挑一行”，但那是未定义行为，别依赖。

In [7]:
# 多列分组：先按地区，再按城市，逐层细分
q('''
SELECT region, city,
       COUNT(*)   AS 订单数,
       SUM(sales) AS 销售额
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY region, city
ORDER BY region, 销售额 DESC
LIMIT 12
''')

,region,city,订单数,销售额
0,华东,上海,507,1702056.00
1,华东,宁波,275,1342969.16
2,华东,杭州,250,1162974.80
3,华东,苏州,265,1060489.59
4,华东,南京,190,822117.45
5,华中,长沙,448,1817177.72
6,华中,郑州,299,1193826.49
7,华中,南昌,215,805753.85
8,华中,合肥,233,779500.78
9,华中,武汉,214,768201.73


> 类比 C++：`GROUP BY region, city` 就像把所有订单塞进一个嵌套的 `map<region, map<city, vector<order>>>`，然后对最内层 vector 做 sum。

## 4. HAVING：对“分组结果”再过滤

`WHERE` 和 `HAVING` 都做过滤，但分工不同：

- **WHERE**：分组**之前**过滤**原始行**（此时还没分组，不能用聚合函数）；
- **HAVING**：分组**之后**过滤**分组结果**（此时能用聚合函数）。

In [8]:
# 需求：找出“销售额超过 150 万”的地区
# 这种“对聚合结果再筛”的条件，只能写在 HAVING 里
q('''
SELECT region,
       SUM(sales) AS 总销售额,
       COUNT(*)   AS 订单数
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY region
HAVING SUM(sales) > 1500000
ORDER BY 总销售额 DESC
''')

,region,总销售额,订单数
0,华东,6090607.00,1487
1,华南,5873532.64,1467
2,华中,5364460.57,1409
3,西南,4051805.89,1103
4,华北,3659920.73,941


In [9]:
# WHERE 和 HAVING 配合：先只看 2024 年以后的订单(WHERE)，
# 再在这些订单里找销售额过 40 万的地区(HAVING)
q('''
SELECT o.region,
       SUM(oi.sales) AS 销售额,
       COUNT(*)      AS 明细行数
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_date >= '2024-01-01'
GROUP BY o.region
HAVING SUM(oi.sales) > 400000
ORDER BY 销售额 DESC
''')

,region,销售额,明细行数
0,华东,3264675.81,771
1,华南,3141435.50,785
2,华中,2699815.44,701
3,西南,2119173.05,567
4,华北,1834504.02,474


> 常见误区：把“销售额 > 100 万”写进 WHERE。因为 WHERE 执行时还没 GROUP BY，根本没有“组销售额”这个东西。**记住：聚合条件永远进 HAVING。**

## 5. SQL 子句执行顺序（彻底讲透）

把完整 SQL 摊开，写法顺序和执行顺序是两回事：

```
写法:  SELECT  DISTINCT  FROM  WHERE  GROUP BY  HAVING  ORDER BY  LIMIT
执行:  FROM  ->  WHERE   ->  GROUP BY -> HAVING -> SELECT -> DISTINCT -> ORDER BY -> LIMIT
```

用两个怪现象解释：

In [10]:
# 怪现象1：WHERE 里不能用 SELECT 刚起的别名（因为 WHERE 比 SELECT 先执行）
try:
    q('''
    SELECT region, AVG(age) AS 人均
    FROM customers
    WHERE 人均 > 40
    GROUP BY region
    ''')
except Exception as e:
    print("如预期报错 ->", type(e).__name__, str(e)[:120])

如预期报错 -> DatabaseError Execution failed on sql '
    SELECT region, AVG(age) AS 人均
    FROM customers
    WHERE 人均 > 40
    GROUP BY region
   


In [11]:
# 怪现象2：但 ORDER BY 里能用别名！因为 ORDER BY 在 SELECT 之后执行
q('''
SELECT region, AVG(age) AS 人均年龄
FROM customers
GROUP BY region
ORDER BY 人均年龄 DESC
''')

,region,人均年龄
0,华南,44.402985
1,华北,43.413043
2,西南,42.142857
3,华中,41.895522
4,华东,39.703125


> 这条执行顺序是 SQL 面试的隐形考点：它能解释 80% 的“为什么我这句报错/结果不对”。把 `FROM→WHERE→GROUP→HAVING→SELECT→ORDER→LIMIT` 这串背下来。

## 6. 分组最常见的报错

最经典的就是“SELECT 里有个非聚合列，却没 GROUP BY 它”。

In [12]:
# 错误写法：SELECT 里有 city，但只 GROUP BY region
# 一个 region 下有多个 city，严格数据库不知道取哪个
try:
    q('''
    SELECT region, city, COUNT(*)
    FROM customers
    GROUP BY region
    ''')
except Exception as e:
    print("如预期 ->", type(e).__name__, str(e)[:160])

> SQLite 这里“宽容”地随便挑了一个 city 返回，但 PostgreSQL 会直接报错。**写的时候按最严格的标准写**：要么把 city 加进 GROUP BY，要么对它套聚合（如 `MAX(city)`），别靠数据库宽松。

## 动手小练习

1. 统计每个 `category`（大类）的商品数量、平均单价、平均成本。
2. 统计每个客户类型 `segment` 的下单客户数（用 COUNT DISTINCT）和总消费额。
3. 找出“订单数 > 500”的配送方式 `ship_mode`。

In [13]:
# TODO(你): 在下面练习
# q(''' ... ''')

## 本课小结

- 聚合函数把多行压一行；无 GROUP BY 时全表是一组。
- `COUNT(*)` 数行、`COUNT(列)` 数非空、`COUNT(DISTINCT 列)` 数去重值，三者别混。
- `GROUP BY` 分组，SELECT 里非聚合列必须都在 GROUP BY 里。
- `WHERE` 分组前滤行、`HAVING` 分组后滤组，聚合条件放 HAVING。
- 执行顺序：FROM→WHERE→GROUP→HAVING→SELECT→ORDER→LIMIT，解释了别名可见性。

## 来源与延伸

- Mode SQL Tutorial · 聚合函数与 GROUP BY：https://mode.com/sql-tutorial/sql-aggregate-functions/
- SQLBolt 第 5 课（SUM/COUNT/GROUP BY）：https://sqlbolt.com/
- SQLZoo · SUM and COUNT：https://sqlzoo.net/wiki/SUM_and_COUNT
- SQLite 官方聚合函数列表：https://www.sqlite.org/lang_aggfunc.html
- LeetCode Database（Easy 多为分组聚合题）：https://leetcode.com/problemset/database/